# 04. Prompt Engineering and Retrieval Augmented Generation

## Aim
To design an effective grounded prompt and implement a Retrieval Augmented Generation (RAG) pipeline.

## Theory
RAG first retrieves relevant information from a knowledge base and then places that information inside a prompt for generation. Grounding the answer in retrieved context helps reduce unsupported statements.

## RAG Pipeline
Documents -> TF-IDF embeddings -> similarity search -> relevant context -> grounded prompt

## Expected Learning
Understand chunking, retrieval, context grounding, and the role of prompt instructions.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    'Python is a high-level programming language used for data science and artificial intelligence.',
    'Artificial Intelligence enables machines to perform tasks that normally require human intelligence.',
    'Machine Learning allows systems to learn patterns from data.',
    'Deep Learning uses neural networks with multiple layers.',
    'Retrieval Augmented Generation combines information retrieval with language generation.',
    'RAG retrieves relevant context before an answer is generated by a language model.'
 ]
vectorizer = TfidfVectorizer(stop_words='english')
document_vectors = vectorizer.fit_transform(documents)

def retrieve_documents(query, count=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors).ravel()
    ranked = np.argsort(scores)[::-1][:count]
    return [(documents[i], scores[i]) for i in ranked if scores[i] > 0]

def build_grounded_prompt(query):
    retrieved = retrieve_documents(query)
    context = '\n'.join(document for document, _ in retrieved)
    prompt = f'''You are a helpful assistant.
Answer only using the context below.
If the answer is unavailable, say: I do not know based on the provided context.

Context:
{context}

Question:
{query}

Answer:
'''
    return prompt, retrieved

query = 'How does RAG reduce hallucination in language models?'
prompt, retrieved = build_grounded_prompt(query)
print('Question:', query)
print('\nRetrieved documents:')
for document, score in retrieved:
    print(f'- score={score:.3f}: {document}')
print('\nGrounded prompt:\n', prompt)
print('RAG pipeline executed successfully.')

Question: How does RAG reduce hallucination in language models?

Retrieved documents:
- score=0.445: RAG retrieves relevant context before an answer is generated by a language model.
- score=0.135: Python is a high-level programming language used for data science and artificial intelligence.

Grounded prompt:
 You are a helpful assistant.
Answer only using the context below.
If the answer is unavailable, say: I do not know based on the provided context.

Context:
RAG retrieves relevant context before an answer is generated by a language model.
Python is a high-level programming language used for data science and artificial intelligence.

Question:
How does RAG reduce hallucination in language models?

Answer:

RAG pipeline executed successfully.


## Observation
The TF-IDF retriever ranks documents according to their similarity to the question. The generated prompt contains only the retrieved context and an instruction to refuse unsupported answers.

## Conclusion
Thus, prompt engineering and a working retrieval-augmented generation pipeline were implemented using a local vector representation.